In [1]:
import os
import base64
import pandas as pd
from tqdm import tqdm
from openai import OpenAI

In [2]:
# 局域网
client = OpenAI(base_url="http://192.168.1.199:1234/v1", api_key="lm-studio")

In [3]:
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')


In [4]:
def process_image(image_path):
    """处理单张图片并返回检测结果"""
    try:
        # 编码图片
        base64_image = encode_image(image_path)

        # 发送请求
        completion = client.chat.completions.create(
            model="qwen2-vl-72b",
            messages=[
                {"role": "system", "content": "你是一个专业的垃圾分类分析助手，请严格按以下要求执行："},
                {"role": "user", "content": [
                    {"type": "text", "text": """基于该图片，请依次回答：
        1. 是否存在有明显边界的生活垃圾堆放（如大量塑料袋/食品包装/纸屑/饮料瓶/家庭废弃物）？[是/否]
        2. 是否存在建筑垃圾（破碎的砖块/水泥块/煤块，排除自然物体如树叶/土壤/树枝/花草）？[是/否]

        要求：
        - 仅用单个汉字回答
        - 不添加任何解释
        - 自然物体不计入建筑垃圾
        - 必须严格区分生活垃圾与建筑垃圾

        示例正确回答格式：
        1. 是
        2. 否"""},
                    {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}}
                ]}
            ],
            temperature=0.1,
            max_tokens=50
        )

        # 解析结果
        response = completion.choices[0].message.content
        living = "Y" if "1. 是" in response else "N"
        construction = "Y" if "2. 是" in response else "N"

        return living, construction

    except Exception as e:
        print(f"处理 {os.path.basename(image_path)} 时出错: {str(e)}")
        return None, None


In [6]:
# 筛选待处理文件：既包括新图片，也包括之前有空值的图片，确保所有文件都正确处理。

# 1. 读取现有DataFrame
base = '/Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/img/'
# folder = 'ZWL'
folder = 'Google'
# folder = 'Faith'

df = pd.read_csv(base + f'cpm__{folder}.csv')
temp_results_path = base +  f"temp_{folder}.csv"  # 临时结果保存路径
output_path = base + f"/vl_{folder}.csv"  # 最终结果路径

# 2. 配置图片文件夹路径
image_folder = base + f'cpm_Selected/{folder}/'  # 图片文件夹路径
supported_ext = ['.jpg', '.jpeg', '.png', '.JPG']  # 支持的图片格式

# 3. 获取需要处理的图片文件列表
image_files = {
    os.path.splitext(f)[0]: os.path.join(image_folder, f)
    for f in os.listdir(image_folder)
    if os.path.splitext(f)[1].lower() in supported_ext
}

# 如果存在临时结果文件，加载已处理的结果
if os.path.exists(temp_results_path):
    temp_results_df = pd.read_csv(temp_results_path)
    processed_files = set(temp_results_df['img_name'])
    
    # 找出需要重新处理的文件（已存在但缺少检测结果）
    missing_results_files = set(
        temp_results_df[temp_results_df[['Domestic', 'Construction']].isnull().any(axis=1)]['img_name']
    )
else:
    temp_results_df = pd.DataFrame(columns=['img_name', 'Domestic', 'Construction'])
    processed_files = set()
    missing_results_files = set()

# 过滤未处理和需要重新处理的图片
image_files_to_process = {
    name: path for name, path in image_files.items()
    if name not in processed_files or name in missing_results_files
}

# 初始化结果存储
results = {
    'img_name': [],  # 存储文件名（不含扩展名）
    'Domestic': [],  # 存储生活垃圾检测结果
    'Construction': []  # 存储建筑垃圾检测结果
}

# 批量处理图片
batch_size = 2  # 每2张图片保存一次
for i, (filename_without_ext, img_path) in enumerate(tqdm(image_files_to_process.items(), desc="正在分析图片")):
    try:
        # 处理图片并获取结果
        living, construction = process_image(img_path)

        # 存储结果
        results['img_name'].append(filename_without_ext)
        results['Domestic'].append(living)
        results['Construction'].append(construction)

        # 每 batch_size 张图片保存一次
        if (i + 1) % batch_size == 0:
            batch_results_df = pd.DataFrame(results)
            temp_results_df = pd.concat([temp_results_df, batch_results_df], ignore_index=True)
            temp_results_df.to_csv(temp_results_path, index=False)
            results = {key: [] for key in results}  # 清空当前批次结果
    except Exception as e:
        print(f"处理 {filename_without_ext} 时出错: {str(e)}")
        continue

# 保存剩余未达到批次大小的结果
if results['img_name']:
    batch_results_df = pd.DataFrame(results)
    temp_results_df = pd.concat([temp_results_df, batch_results_df], ignore_index=True)
    temp_results_df.to_csv(temp_results_path, index=False)

# 将最终结果匹配回原始DF
df = df.merge(temp_results_df, on='img_name', how='left')

# 保存更新后的DF
output_path = base + f"updated_{folder}.csv"
df.to_csv(output_path, index=False)

print(f"处理完成！结果已保存至: {output_path}")



正在分析图片: 100%|█████████████████████████| 594/594 [1:11:40<00:00,  7.24s/it]


处理完成！结果已保存至: /Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/img/updated_Google.csv


In [ ]:
# 这个不读取临时文件中空的


# 1. 读取现有DataFrame
base = '/Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/img/'
# folder = 'ZWL'
folder = 'Google'
# folder = 'Faith'
df = pd.read_csv(base + f'cpm__{folder}.csv')
temp_results_path = base +  f"temp_{folder}.csv"  # 临时结果保存路径
output_path = base + f"/vl_{folder}.csv"  # 最终结果路径


# 2. 配置图片文件夹路径
image_folder = base + f'cpm_Selected/{folder}/'  # 图片文件夹路径
supported_ext = ['.jpg', '.jpeg', '.png', '.JPG']  # 支持的图片格式

# 3. 获取需要处理的图片文件列表
image_files = [
    os.path.join(image_folder, f)
    for f in os.listdir(image_folder)
    if os.path.splitext(f)[1].lower() in supported_ext
]

# 如果存在临时结果文件，加载已处理的结果
if os.path.exists(temp_results_path):
    temp_results_df = pd.read_csv(temp_results_path)
    processed_files = set(temp_results_df['img_name'])
else:
    temp_results_df = pd.DataFrame(columns=['img_name', 'Domestic', 'Construction'])
    processed_files = set()

# 过滤未处理的图片
image_files = [
    img_path for img_path in image_files
    if os.path.splitext(os.path.basename(img_path))[0] not in processed_files
]

# 初始化结果存储
results = {
    'img_name': [],  # 存储文件名（不含扩展名）
    'Domestic': [],  # 存储生活垃圾检测结果
    'Construction': []  # 存储建筑垃圾检测结果
}

# 批量处理图片
batch_size = 2  # 每2张图片保存一次
for i, img_path in enumerate(tqdm(image_files, desc="正在分析图片")):
    try:
        # 获取文件名（不含路径和扩展名）
        filename = os.path.basename(img_path)
        filename_without_ext = os.path.splitext(filename)[0]

        # 处理图片并获取结果
        living, construction = process_image(img_path)

        # 存储结果
        results['img_name'].append(filename_without_ext)
        results['Domestic'].append(living)
        results['Construction'].append(construction)

        # 每200张图片保存一次
        if (i + 1) % batch_size == 0:
            # 将当前批次结果保存到临时文件
            batch_results_df = pd.DataFrame(results)
            temp_results_df = pd.concat([temp_results_df, batch_results_df], ignore_index=True)
            temp_results_df.to_csv(temp_results_path, index=False)
            # 清空当前批次结果
            results = {key: [] for key in results}

    except Exception as e:
        print(f"处理 {filename} 时出错: {str(e)}")
        continue

# 保存剩余未达到批次大小的结果
if results['img_name']:
    batch_results_df = pd.DataFrame(results)
    temp_results_df = pd.concat([temp_results_df, batch_results_df], ignore_index=True)
    temp_results_df.to_csv(temp_results_path, index=False)


# 将最终结果匹配回原始DF
temp_results_df = temp_results_df.drop_duplicates()
df = df.merge(temp_results_df, on='img_name', how='left')

# 保存更新后的DF
df.to_csv(output_path, index=False)

print(f"处理完成！结果已保存至: {output_path}")

# Test

In [ ]:
# Text Example

completion = client.chat.completions.create(
  model="qwen2-vl-72b",
  messages=[
    {"role": "system", "content": "Always answer in rhymes."},
    {"role": "user", "content": "Introduce yourself."}
  ],
  temperature=0.7,
)

print(completion.choices[0].message)


In [ ]:
# Single image example

base64_image = encode_image("/Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/img/cpm_Selected/Google/_4lITCW2mM3nUENO9mpWag_270.jpg")  # 替换为你的图片路径
# print(base64_image)

completion = client.chat.completions.create(
  model="qwen2-vl-72b",
  messages=[
    {"role": "system", "content": "你是一个专业的垃圾分类分析助手，请严格按以下要求执行："},
    {"role": "user", "content": [
        {"type": "text", "text": """基于该图片，请依次回答：
        1. 是否存在有明显边界的生活垃圾堆放（如大量塑料袋/食品包装/纸屑/饮料瓶/家庭废弃物）？[是/否]
        2. 是否存在建筑垃圾（破碎的砖块/水泥块/煤块，排除自然物体如树叶/土壤/树枝/花草）？[是/否]

        要求：
        - 仅用单个汉字回答
        - 不添加任何解释
        - 自然物体不计入建筑垃圾
        - 必须严格区分生活垃圾与建筑垃圾

        示例正确回答格式：
        1. 是
        2. 否"""},
        {"type": "image_url",
         "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}
        }
    ]}
  ],
  temperature=0.1,  # 降低随机性保证确定性
  max_tokens=50
)

print(completion.choices[0].message.content)

In [ ]:
# 测试遥感图像

import base64
from openai import OpenAI

client = OpenAI(base_url="http://192.168.1.199:1234/v1", api_key="lm-studio")

def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

base64_image = encode_image("/Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/RS_Waste/VOC2012/train/JPEGImages/10002.jpg")


completion = client.chat.completions.create(
  model="qwen2-vl-72b",
  messages=[
    {"role": "system", "content": "你是一个专业的垃圾分类分析助手，请严格按以下要求执行："},
    {"role": "user", "content": [
        {"type": "text", "text": """
[输入规格]
- 图像尺寸: 1024x1024像素
- 通道要求: RGB三通道
- 空间分辨率: 请指定(例如0.3m/px)

[检测目标]
垃圾堆(Garbage dump)定义标准:
1. 形态特征:
   - 非结构化聚集(>100像素不规则团块)
   - 纹理对比度>60%(局部标准差)
2. 光谱特征:
   - 有机质区域(R均值<100, G均值<110, B均值<90)
   - 金属反光区(R通道值>200的像素占比>5%
3. 环境特征:
   - 500像素范围内存在道路(线状特征)
   - 300像素内有车辆/机械痕迹

[处理流程]
1. 预处理:
   - CLAHE对比度增强(clip=3.0, tile=8x8)
   - 高斯滤波去噪(σ=1.5)
2. 特征提取:
   - ResNet50多尺度特征金字塔
   - 注意力机制权重分配:
     - 空间注意力: 0.6
     - 通道注意力: 0.4
3. 后处理:
   - 面积过滤: 最小1000像素(约0.09公顷@0.3m/px)
   - NMS阈值: IoU 0.35

[输出规范]
JSON格式包含:
- 边界框坐标([xmin,ymin,xmax,ymax])
- 置信度(0-1)
- 垃圾堆类型(建筑/生活/工业)
- 估计面积(平方米)
"""},
        {"type": "image_url",
         "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}
        }
    ]}
  ],
  temperature=0.1,  # 降低随机性保证确定性
  max_tokens=50
)

print(completion.choices[0].message)